# Bayesian Linear Regression

Companion notebook for the [Bayesian Linear Regression lesson](https://ml-viz-ruby.vercel.app/courses/bayesian-methods/01-bayesian-linear-regression).

We compute the Gaussian **posterior over weights** in closed form, confirm its mean equals **ridge
regression**, and plot the **predictive distribution** — watching the error bars widen away from the
data. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

## 1 — Data and design matrix

A few noisy points from a linear function, clustered on the left so we can see what happens where
there is no data. Features are [1, x] (bias + slope).

In [ ]:
x = np.array([-2.0, -1.6, -1.2, -0.8, -0.4, 0.0])     # data only on the left
y = 0.9 * x + 0.3 + rng.normal(0, 0.15, size=x.shape)
Phi = np.c_[np.ones_like(x), x]                        # design matrix [1, x]
alpha, beta = 2.0, 25.0                                 # prior precision, noise precision

## 2 — The closed-form Gaussian posterior over weights

S_N^{-1} = αI + β ΦᵀΦ ;  m_N = β S_N Φᵀy. With Gaussian prior and likelihood the posterior is
Gaussian — no sampling needed.

In [ ]:
def posterior(Phi, y, alpha, beta):
    d = Phi.shape[1]
    S_N_inv = alpha * np.eye(d) + beta * Phi.T @ Phi
    S_N = np.linalg.inv(S_N_inv)
    m_N = beta * S_N @ Phi.T @ y
    return m_N, S_N

m_N, S_N = posterior(Phi, y, alpha, beta)
print('posterior mean weights [bias, slope]:', m_N.round(3))
print('posterior covariance:\n', S_N.round(4))

## 3 — The posterior mean IS ridge regression

Ridge with λ = α/β should reproduce the posterior mean exactly.

In [ ]:
lam = alpha / beta
ridge_w = np.linalg.solve(Phi.T @ Phi + lam * np.eye(2), Phi.T @ y)
print('ridge (λ=α/β) weights: ', ridge_w.round(3))
print('posterior mean weights:', m_N.round(3))
assert np.allclose(ridge_w, m_N)
print('\u2713 posterior mean equals the ridge solution')

## 4 — Predictive distribution: error bars widen away from data

σ²(x*) = β⁻¹ + φ(x*)ᵀ S_N φ(x*). We plot the predictive mean ± 2σ across a range. Notice the band
is tight over the data (left) and flares out where we extrapolate (right).

In [ ]:
xs = np.linspace(-3, 3, 200)
Phis = np.c_[np.ones_like(xs), xs]
mean = Phis @ m_N
var = 1.0 / beta + np.einsum('ij,jk,ik->i', Phis, S_N, Phis)
sd = np.sqrt(var)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.fill_between(xs, mean - 2*sd, mean + 2*sd, color='#6366f1', alpha=0.3, label='±2σ predictive')
ax.plot(xs, mean, color='#818cf8', label='predictive mean')
ax.scatter(x, y, color='#2dd4bf', zorder=5, label='data')
ax.axvspan(x.min(), x.max(), color='#2dd4bf', alpha=0.05)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Predictive uncertainty is tight on the data, flares out in extrapolation')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'predictive sd at x=0 (in data):  {np.sqrt(1/beta + np.r_[1,0]@S_N@np.r_[1,0]):.3f}')
print(f'predictive sd at x=3 (far away):  {np.sqrt(1/beta + np.r_[1,3]@S_N@np.r_[1,3]):.3f}')

## ✏️ Your turn

**Exercise.** Implement `predictive(phi_star, m_N, S_N, beta)` returning the `(mean, variance)` of the
predictive distribution at a single input feature vector `phi_star`:
mean = m_Nᵀφ*, variance = β⁻¹ + φ*ᵀ S_N φ*.

In [ ]:
def predictive(phi_star, m_N, S_N, beta):
    # TODO(you): return (predictive mean, predictive variance) at phi_star
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
m0, v0 = predictive(np.array([1.0, 0.0]), m_N, S_N, beta)   # x=0, inside the data
m3, v3 = predictive(np.array([1.0, 3.0]), m_N, S_N, beta)   # x=3, far away
assert np.isclose(m0, m_N[0])                                # mean at x=0 is the bias
assert v3 > v0                                               # uncertainty grows away from data
assert v0 > 1.0 / beta                                       # always at least the noise floor
print(f'\u2713 predictive correct: var(x=0)={v0:.3f} < var(x=3)={v3:.3f}')

<details>
<summary>Solution</summary>

```python
def predictive(phi_star, m_N, S_N, beta):
    mean = phi_star @ m_N
    var = 1.0 / beta + phi_star @ S_N @ phi_star
    return mean, var
```

The variance is the noise floor β⁻¹ plus a model-uncertainty term φ*ᵀS_Nφ* that grows as φ* moves
into regions the posterior is unsure about — which is exactly where there's little data.

</details>